#  Les lignes de courant

À un instant donné, en chaque point de l'espace, le vent est un **vecteur** $\vec{V}=(u,w)$ (ici dans le plan vertical $(x,z)$). Une **ligne de courant** (*streamline*) est une courbe qui est, partout, **tangente à ce vecteur vitesse**.

Tracer des lignes tangentes au champ « à la main » serait pénible. Il existe un **raccourci mathématique** très puissant, à une condition.

**La condition : un écoulement incompressible (sans création ni perte de matière).** En 2D, cela s'écrit « la divergence est nulle » :
$$\frac{\partial u}{\partial x} + \frac{\partial w}{\partial z} = 0.$$

**Le théorème.** Quand cette condition est vraie, il existe une **fonction scalaire** $\psi(x,z)$ — un simple nombre attaché à chaque point — telle que les deux composantes de la vitesse sont ses dérivées :
$$u = -\frac{\partial \psi}{\partial z}, \qquad w = +\frac{\partial \psi}{\partial x}.$$
Cette $\psi$ s'appelle la **fonction de courant**.

 On peut vérifier que le long d'une courbe où $\psi$ reste constant, le déplacement est exactement tangent à $\vec V$. Autrement dit :
$$\boxed{\text{les lignes de courant} = \text{les lignes de niveau (isolignes) de } \psi.}$$
Tracer les lignes de courant revient donc à tracer un simple **contour** de $\psi$ (comme les courbes de niveau d'une carte topographique). Un `plt.contour(psi)` suffit.

Bonus : la **différence de $\psi$** entre deux lignes donne le **débit** qui passe entre elles. Resserrement des isolignes ⇔ écoulement intense.

---
## Étape 3 — Le cas de l'atmosphère : fonction de courant **de masse**

L'air n'est pas incompressible : sa densité $\rho_0(z)$ **chute avec l'altitude** (l'air est plus dense en bas). La divergence de la vitesse n'est donc pas nulle.

**La bonne quantité conservée** n'est pas le volume mais la **masse**. L'approximation *anélastique* (standard en convection) dit que c'est le **flux de masse** $(\rho_0 u,\ \rho_0 w)$ qui est sans divergence :
$$\frac{\partial (\rho_0\, u)}{\partial x} + \frac{\partial (\rho_0\, w)}{\partial z} = 0.$$
On définit alors une **fonction de courant de masse** $\psi$ par :
$$\rho_0\, u = -\frac{\partial \psi}{\partial z}, \qquad \rho_0\, w = +\frac{\partial \psi}{\partial x}.$$
Ses isolignes sont les lignes de courant du **transport de masse** — exactement ce qui décrit la circulation de retournement (*overturning*) de la convection.

---
## Étape 4 — Pourquoi pas une simple intégration ? (le piège)

Naïvement, on pourrait obtenir $\psi$ en **intégrant** : par exemple, partir de $w = \frac{1}{\rho_0}\partial_x\psi$ et cumuler $\psi(x,z) = \int_0^x \rho_0\, w\,dx'$.

**Le problème :** cette recette suppose que la condition de divergence nulle est *parfaitement* vérifiée. Or nos données réelles ne le sont jamais exactement : on a moyenné sur un $y$ de taille finie, il y a du bruit turbulent, et la grille est discrète. Du coup :
- intégrer **horizontalement** ($\int \rho_0 w\, dx$) ou **verticalement** ($-\int \rho_0 u\, dz$) donne **deux résultats différents** ;
- le résultat **dépend du chemin** suivi pour intégrer. Ce n'est pas acceptable.

---
## Étape 5 — La solution propre : décomposition de Helmholtz + équation de Poisson

Tout champ de vecteurs se décompose de façon unique (théorème de **Helmholtz**) en deux morceaux :
$$\text{flux de masse} = \underbrace{\text{partie rotationnelle}}_{\text{tourne, porte la circulation}} + \underbrace{\text{partie divergente}}_{\text{source/puits, pas de circulation}}.$$
Seule la **partie rotationnelle** correspond à une vraie fonction de courant. On veut donc l'extraire proprement et **jeter** le reste.

Pour cela on introduit la **vorticité** (le « taux de rotation local » de l'écoulement). En appliquant l'opérateur rotationnel à la définition de $\psi$, les deux relations $\rho_0 u=-\partial_z\psi$ et $\rho_0 w=+\partial_x\psi$ se combinent en **une seule équation** :
$$\boxed{\nabla^2\psi \;=\; \frac{\partial(\rho_0 w)}{\partial x} - \frac{\partial(\rho_0 u)}{\partial z}}$$
où $\nabla^2 = \partial_x^2 + \partial_z^2$ est le **Laplacien**. C'est une **équation de Poisson** : le membre de droite (calculable directement à partir des données $u,w$) est connu, et on cherche $\psi$.

**Comment on la résout numériquement ?**
1. On calcule le membre de droite (la vorticité de masse) par différences finies.
2. On écrit le Laplacien comme une grande **matrice creuse** (chaque point relié à ses 4 voisins).
3. On impose une **condition au bord** : ici $\psi=0$ sur le contour du domaine (Dirichlet). *(À adapter en périodique si le domaine l'est — voir remarque en fin de notebook.)*
4. On résout le système linéaire $A\,\psi = b$ en une fois (`scipy.sparse.linalg.spsolve`).

**Le résultat** est un $\psi$ **unique** et **indépendant du chemin** : la résolution projette automatiquement le champ sur sa partie rotationnelle et laisse de côté la partie divergente.

---
###  Résumé
> Une ligne de courant est une isoligne de la fonction de courant $\psi$ ; en atmosphère on travaille avec $\psi$ **de masse** ($\rho_0 u=-\partial_z\psi$, $\rho_0 w=+\partial_x\psi$) ; et pour l'obtenir proprement sur des données réelles on **résout une équation de Poisson** $\nabla^2\psi=\partial_x(\rho_0 w)-\partial_z(\rho_0 u)$ plutôt que d'intégrer naïvement.

## 0. Imports & configuration

Calé sur le pipeline **large300** : fichiers 3D séparés par variable dans `3D/MESONH_RCE_large300_3D_{var}.nc` (variables `ua`,`va`,`wa`,`ta`,`pa`,`hus`), altitude reprise du 1D, état stationnaire = **dernier tiers** des pas de temps, $\rho_0(z)$ calculé via la température virtuelle. Lecture **niveau par niveau** pour la RAM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.25,
    'image.cmap': 'RdBu_r',
})

# ============================================================
#  CONFIGURATION  (identique au notebook rcemip_large300)
# ============================================================
DIR_3D = '3D';  DIR_2D = '2D';  DIR_1D = '1D'
def path3d(var): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')
def path2d(v): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{v}.nc')
def path1d(var): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{var}.nc')

Rd, Rv = 287.05, 461.5
EPSILON = Rd / Rv          # ≈ 0.622
BLOC    = 2                # taille de bloc temporel (RAM)

USE_SYNTHETIC = False      # True = champ jouet (démo sans données) ; False = large300

# Résolution horizontale large300 (mêmes valeurs que le pipeline rcemip).
# Forcée à 2000 m si les coordonnées sont des indices entiers.
DX_DEFAULT = 1000.0        # m

print('Config prête. USE_SYNTHETIC =', USE_SYNTHETIC)

## 1. Chargement large300 : coupe $(x,z)$ moyennée sur $y$ et sur le temps

On construit la coupe 2D $(z,x)$ en moyennant **sur $y$** (révèle l'overturning) **et sur le temps** (état stationnaire). En parallèle on accumule $\langle u'w'\rangle$ moyenné de la même façon, avec la **même convention d'anomalie spatiale** $(y,x)$ par niveau que ton pipeline.

Trois étapes : (1a) métadonnées + altitude, (1b) profil $\rho_0(z)$ via $T_v$, (1c) coupes moyennées niveau par niveau.

In [ ]:
# --- (1a) Métadonnées : dims, tailles, altitude, fenêtre stationnaire ---
if not USE_SYNTHETIC:
    _ds = xr.open_dataset(path3d('ua'));  _da = _ds['ua']
    dim_t, dim_z, dim_y, dim_x = _da.dims        # ordre (t, z, y, x)
    n_t = _da.sizes[dim_t];  n_z = _da.sizes[dim_z]
    n_y = _da.sizes[dim_y];  n_x = _da.sizes[dim_x]
    _ds.close();  del _ds, _da;  gc.collect()

    # altitude depuis le fichier 1D (plus fiable que la coord brute)
    _t1 = xr.open_dataset(path1d('ua_avg'))
    alt = _t1['altitude'].values.astype(float).copy()
    _t1.close()

    t_stat   = int(2 * n_t / 3)          # dernier tiers = stationnaire
    idx_stat = slice(t_stat, None)
    n_stat   = n_t - t_stat
    print(f'Grille : {n_t} t × {n_z} z × {n_y} y × {n_x} x')
    print(f'Altitude : {alt[0]:.0f} → {alt[-1]:.0f} m')
    print(f'Stationnaire : t={t_stat}→{n_t-1} ({n_stat} pas)')

In [ ]:
# --- (1b) Profil rho_0(z) via température virtuelle (gaz parfaits) ---
if not USE_SYNTHETIC:
    rho0_sum = np.zeros(n_z);  n_rho = 0
    ds_ta  = xr.open_dataset(path3d('ta'))
    ds_pa  = xr.open_dataset(path3d('pa'))
    ds_hus = xr.open_dataset(path3d('hus'))
    for t0 in range(t_stat, n_t, BLOC):
        t1 = min(t0 + BLOC, n_t);  sl = {dim_t: slice(t0, t1)}
        T_blk  = ds_ta['ta'].isel(sl).values
        p_blk  = ds_pa['pa'].isel(sl).values
        qv_blk = ds_hus['hus'].isel(sl).values
        Tv_blk  = T_blk * (1.0 + qv_blk/EPSILON) / (1.0 + qv_blk)
        rho_blk = p_blk / (Rd * Tv_blk)
        rho0_sum += rho_blk.mean(axis=(0, 2, 3)) * (t1 - t0)
        n_rho    += (t1 - t0)
        del T_blk, p_blk, qv_blk, Tv_blk, rho_blk;  gc.collect()
    ds_ta.close();  ds_pa.close();  ds_hus.close()
    del ds_ta, ds_pa, ds_hus;  gc.collect()
    rho0 = rho0_sum / n_rho
    print(f'ρ₀ : surface {rho0[0]:.3f} → sommet {rho0[-1]:.4f} kg/m³ '
          f'({n_rho} pas)')

In [ ]:
# --- (1c) Coupes (z,x) moyennées sur y ET sur le temps stationnaire ---
#   U, W      : <.>_{y,t}                      -> pour psi (lignes de courant)
#   UPWP      : <u'w'>_{y,t}  (anomalie (y,x)) -> flux de Reynolds local (z,x)
#   ubar,wbar : <.>_{x,y,t}                    -> profils pour diagnostics
#   >>> On garde TOUS les niveaux z (aucune troncature verticale). <<<
if not USE_SYNTHETIC:
    z = alt.copy()
    # --- résolution horizontale : lue depuis les coords, sinon DX_DEFAULT ---
    _dsx = xr.open_dataset(path3d('ua'))
    _xv = _dsx['ua'][dim_x].values
    print(_xv)
    dx = float(np.mean(np.diff(_xv)))
    _dsx.close(); del _dsx, _xv
    if abs(dx) <= 1.5:           # coords = indices entiers -> vraie résolution
        dx = DX_DEFAULT
    x = np.arange(n_x, dtype=float) * dx        # axe x en MÈTRES
    print(f'Résolution horizontale dx = {dx:.0f} m  (domaine {n_x*dx/1000:.0f} km)')
    U  = np.zeros((n_z, n_x));  W = np.zeros((n_z, n_x))
    UPWP = np.zeros((n_z, n_x))
    ubar = np.zeros(n_z);  wbar = np.zeros(n_z)

    # niveaux cibles où l'on garde une tranche 2D (y,x) pour le co-spectre
    Z_TARGETS = [5000., 10000., 15000.]   # m
    slices2d = {}                          # iz -> (u_p2d, w_p2d) moyennés en t
    iz_targets = [int(np.argmin(np.abs(alt - zt))) for zt in Z_TARGETS]

    ds_u = xr.open_dataset(path3d('ua'))
    ds_w = xr.open_dataset(path3d('wa'))
    for iz in range(n_z):
        sl = {dim_t: idx_stat, dim_z: iz}
        u_lev = ds_u['ua'].isel(sl).values     # (n_stat, ny, nx)
        w_lev = ds_w['wa'].isel(sl).values
        # anomalie spatiale (y,x) par pas de temps -> convention pipeline
        u_p = u_lev - u_lev.mean(axis=(1, 2), keepdims=True)
        w_p = w_lev - w_lev.mean(axis=(1, 2), keepdims=True)
        # moyennes <.>_{y,t} -> profils en x
        U[iz]    = u_lev.mean(axis=(0, 1))      # moyenne sur t puis y
        W[iz]    = w_lev.mean(axis=(0, 1))
        UPWP[iz] = (u_p * w_p).mean(axis=(0, 1))
        ubar[iz] = u_lev.mean();  wbar[iz] = w_lev.mean()
        if iz in iz_targets:                 # tranche (y,x) moyennée sur t
            slices2d[iz] = (u_p.mean(axis=0), w_p.mean(axis=0))
        del u_lev, w_lev, u_p, w_p;  gc.collect()
        if iz % 10 == 0: print(f'  niveau {iz}/{n_z-1} ({alt[iz]:.0f} m)')
    ds_u.close();  ds_w.close();  del ds_u, ds_w;  gc.collect()

    upwp = UPWP                               # alias attendu par la suite
    print('Coupes (z,x) prêtes :', U.shape)
    # >>> AUCUNE troncature verticale : on conserve tous les z. <<<
    n_z = z.size
    print(f'Tous les niveaux conservés : {z[0]:.0f}–{z[-1]:.0f} m  ({n_z} niveaux)')


In [ ]:
# --- Repli synthétique (uniquement si USE_SYNTHETIC=True) : champ jouet ---
#     overturning sinusoïdal + panaches + bruit. Sert à exécuter le notebook
#     sans les NetCDF. NE PAS interpréter physiquement.
if USE_SYNTHETIC:
    rng = np.random.default_rng(0)
    n_z, n_x = 74, 256
    z = np.linspace(0, 20000, n_z);  x = np.linspace(0, 300000, n_x)
    XX, ZZ = np.meshgrid(x, z);  rho0 = 1.2*np.exp(-z/8000.)
    H, L = z.max(), x.max()
    psi0 = 4e3*np.sin(np.pi*ZZ/H)*np.sin(2*np.pi*XX/L)
    dz = z[1]-z[0];  dx = x[1]-x[0]
    W = np.gradient(psi0, dx, axis=1)/rho0[:,None]
    U = -np.gradient(psi0, dz, axis=0)/rho0[:,None]
    for xc in rng.uniform(0, L, 6):
        W += rng.uniform(3,8)*np.exp(-((XX-xc)/6000.)**2)*np.sin(np.pi*ZZ/H)
    U += 0.5*rng.standard_normal(U.shape);  W += 0.3*rng.standard_normal(W.shape)
    ubar = U.mean(axis=1);  wbar = W.mean(axis=1)
    upwp = (U-ubar[:,None])*(W-wbar[:,None])
    print('⚠️ MODE SYNTHÉTIQUE — données fabriquées, non physiques.')
    print('Grille (nz,nx) =', U.shape)

In [ ]:
# --- masques humide/sec via PRW ---
ds_prw  = xr.open_dataset(path2d('prw'))
prw_all = ds_prw['prw'].load(); ds_prw.close()
prw_mean  = prw_all.isel({dim_t: slice(t_stat,None)}).mean(dim=dim_t)
PRW_SEUIL = float(np.median(prw_all.values.ravel()))
mh = (prw_mean.values > PRW_SEUIL); ms = ~mh
f_h, f_s = mh.mean(), ms.mean()
del prw_all; gc.collect()
print(f'Humide {f_h*100:.1f}%   Sec {f_s*100:.1f}%')

## 1bis. Fonction de courant de masse $\psi$ — résolution propre par Poisson

**Chaîne logique.** En anélastique, la continuité sur le champ moyenné en $y$ s'écrit
$$\partial_x(\rho_0\bar u) + \partial_z(\rho_0\bar w) = 0,$$
c.-à-d. le couple flux de masse $(\rho_0\bar u,\,\rho_0\bar w)$ est à **divergence nulle**. Cette nullité garantit l'existence d'une fonction de courant $\psi$ telle que
$$\rho_0\,\bar u = -\partial_z\psi,\qquad \rho_0\,\bar w = +\partial_x\psi.$$
Les **isolignes de $\psi$ sont les lignes de courant** (le flux de masse leur est tangent).

**Pourquoi pas une simple intégration ?** Le champ réel (moyenné sur un $y$ fini, bruité, discret) n'est *jamais* exactement non-divergent. Intégrer $\rho_0\bar w$ en $x$ ou $-\rho_0\bar u$ en $z$ donne alors des résultats **dépendants du chemin**. Il faut extraire proprement la seule partie qui porte une circulation.

**Décomposition de Helmholtz + Poisson.** On scinde le flux de masse en partie rotationnelle (la circulation) + partie divergente (le résidu, qu'on écarte). En prenant le rotationnel de la définition de $\psi$ on obtient une **équation de Poisson** :
$$\nabla^2\psi = \partial_x(\rho_0\bar w) - \partial_z(\rho_0\bar u) \;\equiv\; \omega_{\text{masse}},$$
avec des **conditions de Dirichlet** en $z$ : $\psi=0$ en **bas** ($z=0$, aucun flux ne traverse le sol) et $\psi_{\text{top}}=\int_0^{z_{\text{top}}}\rho_0(z)\,\bar u(z)\,\mathrm{d}z$ en **haut** (transport de masse net intégré sous le sommet), et des **bords gauche/droite périodiques** en $x$. Sa solution est **unique, indépendante du chemin**, et projette automatiquement hors de la partie divergente.

> **Note de signe.** La définition $\rho_0\bar u=-\partial_z\psi$ donne formellement $\psi(z_{\text{top}})-\psi(0)=-\int_0^{z_{\text{top}}}\rho_0\bar u\,\mathrm{d}z$. Le code applique la convention demandée $\psi_{\text{top}}=+\int_0^{z_{\text{top}}}\rho_0\bar u\,\mathrm{d}z$ (signe global de $\psi$ inversé, sans effet sur la forme des lignes de courant).

## 2-3. Visualisation : lignes de courant + faible corrélation $u'w'$

On superpose :
- les **isolignes de $\psi$** (lignes de courant de masse) ;
- en fond couleur, le **flux de Reynolds local** $u'w'$.

In [ ]:
# ======================================================================
#  Coupes (z,x) à des instants t donnés — moyennées sur y SEULEMENT
#  Version autonome : ne dépend que de dim_t, dim_z, dim_x, n_t, n_x,
#  alt, t_stat, n_stat, x, path3d. Recalcule rho0 et tout le reste.
#  >>> Tous les niveaux z sont conservés (aucune troncature). <<<
# ======================================================================
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve

T_SNAPS = [t_stat, t_stat + n_stat//2, n_t - 1]   # instants à tracer

# --- rho0 sur la grille COMPLETE ---
_nz_full = alt.size
rho0_full = np.zeros(_nz_full); _nr = 0
_da = xr.open_dataset(path3d('ta')); _dp = xr.open_dataset(path3d('pa')); _dh = xr.open_dataset(path3d('hus'))
for t0 in range(t_stat, n_t, BLOC):
    t1 = min(t0+BLOC, n_t); sl = {dim_t: slice(t0,t1)}
    T = _da['ta'].isel(sl).values; P = _dp['pa'].isel(sl).values; Q = _dh['hus'].isel(sl).values
    Tv = T*(1.0+Q/EPSILON)/(1.0+Q)
    rho0_full += (P/(Rd*Tv)).mean(axis=(0,2,3))*(t1-t0); _nr += (t1-t0)
    del T,P,Q,Tv; gc.collect()
_da.close(); _dp.close(); _dh.close(); del _da,_dp,_dh; gc.collect()
rho0_full /= _nr

def coupe_instant(it):
    """Coupe (z,x) moyennée sur y à l'instant it — TOUS les niveaux z."""
    U_i = np.zeros((_nz_full, n_x)); W_i = np.zeros((_nz_full, n_x)); UP_i = np.zeros((_nz_full, n_x))
    du = xr.open_dataset(path3d('ua')); dw = xr.open_dataset(path3d('wa'))
    for iz in range(_nz_full):
        u_yx = du['ua'].isel({dim_t: it, dim_z: iz}).values
        w_yx = dw['wa'].isel({dim_t: it, dim_z: iz}).values
        u_p = u_yx - u_yx.mean(); w_p = w_yx - w_yx.mean()
        U_i[iz] = u_yx.mean(axis=0); W_i[iz] = w_yx.mean(axis=0)
        UP_i[iz] = (u_p*w_p).mean(axis=0)
        del u_yx, w_yx, u_p, w_p
    du.close(); dw.close(); del du, dw; gc.collect()
    return alt, rho0_full, U_i, W_i, UP_i

def psi_poisson_periodic(U_i, W_i, x, z_i, rho_i):
    """psi de masse : PÉRIODIQUE en x, DIRICHLET INHOMOGÈNE en z.
       bas  : psi(z=0)    = 0
       haut : psi(z=ztop) = ∫_0^ztop rho0(z) <u>(z) dz   (transport de masse net)
    On résout le Poisson sur les niveaux INTÉRIEURS ; les deux niveaux de bord
    (0 et nz-1) sont imposés et passent au second membre.
    """
    nz, nx = U_i.shape
    dxl = float(x[1]-x[0]); dzl = float(z_i[1]-z_i[0])
    rhoU = rho_i[:, None]*U_i
    rhoW = rho_i[:, None]*W_i

    # --- terme source : rotationnel du flux de masse ---
    #     dérivée en x PÉRIODIQUE (np.roll fait le wrap)
    ddx_rhoW = (np.roll(rhoW, -1, axis=1) - np.roll(rhoW, 1, axis=1)) / (2*dxl)
    ddz_rhoU = np.gradient(rhoU, dzl, axis=0)
    omega = ddx_rhoW - ddz_rhoU

    # --- valeurs de Dirichlet aux bords haut/bas (constantes en x) ---
    rho0_ubar = (rho_i * U_i.mean(axis=1))          # rho0(z) * <u>_x(z)  (nz,)
    psi_bot = 0.0
    # ∫_0^ztop rho0 <u> dz  (trapèze maison -> robuste NumPy 1.x/2.x)
    psi_top = float(np.sum(0.5*(rho0_ubar[1:]+rho0_ubar[:-1])*np.diff(z_i)))
    print(f"    psi_bot = {psi_bot:+.3e}   psi_top = {psi_top:+.3e}  (kg m^-1 s^-1)")

    # --- Lx périodique : toutes les nx colonnes sont inconnues ---
    Lx = diags([1., -2., 1.], [-1, 0, 1], shape=(nx, nx), format='lil')
    Lx[0, nx-1] = 1.0       # le point 0 voit le point nx-1 (wrap gauche)
    Lx[nx-1, 0] = 1.0       # le point nx-1 voit le point 0 (wrap droit)
    Lx = Lx.tocsr() / dxl**2

    # --- Lz Dirichlet : seulement les nz-2 niveaux intérieurs ---
    Lz = diags([1., -2., 1.], [-1, 0, 1], shape=(nz-2, nz-2)).tocsr() / dzl**2

    Ix = identity(nx)
    Iz = identity(nz-2)
    A = (kron(Iz, Lx) + kron(Lz, Ix)).tocsr()

    # --- second membre : source intérieure + report des bords Dirichlet ---
    b = omega[1:-1, :].copy()                # (nz-2, nx)
    # le niveau intérieur le plus bas (j=0) a un voisin imposé psi_bot
    b[0,  :] -= psi_bot / dzl**2
    # le niveau intérieur le plus haut (j=nz-3) a un voisin imposé psi_top
    b[-1, :] -= psi_top / dzl**2
    b = b.ravel()

    psi = np.zeros((nz, nx))
    psi[0,  :] = psi_bot
    psi[-1, :] = psi_top
    psi[1:-1, :] = spsolve(A, b).reshape(nz-2, nx)
    return psi


for it in T_SNAPS:
    z_i, rho_i, U_i, W_i, UP_i = coupe_instant(it)
    psi_i = psi_poisson_periodic(U_i, W_i, x, z_i, rho_i)
    Xkm, Zkm = x/1000, z_i/1000

    fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    ax = axes[0]
    wmax = np.percentile(np.abs(W_i), 99) + 1e-12
    pc = ax.pcolormesh(Xkm, Zkm, W_i, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-wmax, vmax=wmax), cmap='RdBu_r')
    ax.contour(Xkm, Zkm, psi_i, levels=30, colors='k', linewidths=0.6)
    ax.set_title(f"(a) Lignes de courant (psi) + <w>_y  -  it = {it}")
    ax.set_ylabel('z [km]'); fig.colorbar(pc, ax=ax, label='<w>_y [m/s]', pad=0.01)

    ax = axes[1]
    fmax = np.percentile(np.abs(UP_i), 98) + 1e-12
    pc = ax.pcolormesh(Xkm, Zkm, UP_i, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-fmax, vmax=fmax), cmap='RdBu_r')
    ax.contour(Xkm, Zkm, psi_i, levels=30, colors='k', linewidths=0.6)
    ax.set_title("(b) Memes lignes de courant + <u'w'>_y")
    ax.set_xlabel('x [km]'); ax.set_ylabel('z [km]')
    fig.colorbar(pc, ax=ax, label="<u'w'>_y [m2/s2]", pad=0.01)
    plt.tight_layout(); plt.show()


In [ ]:
# ======================================================================
#  Coupes (z,x) à des instants t donnés — moyennées sur y SEULEMENT
#  Version autonome : ne dépend que de dim_t, dim_z, dim_x, n_t, n_x,
#  alt, t_stat, n_stat, x, path3d. Recalcule rho0 et tout le reste.
#  >>> Tous les niveaux z sont conservés (aucune troncature). <<<
# ======================================================================
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve

T_SNAPS = list(range(t_stat, n_t, 5))   # instants à tracer

# --- rho0 sur la grille COMPLETE ---
_nz_full = alt.size
rho0_full = np.zeros(_nz_full); _nr = 0
_da = xr.open_dataset(path3d('ta')); _dp = xr.open_dataset(path3d('pa')); _dh = xr.open_dataset(path3d('hus'))
for t0 in range(t_stat, n_t, BLOC):
    t1 = min(t0+BLOC, n_t); sl = {dim_t: slice(t0,t1)}
    T = _da['ta'].isel(sl).values; P = _dp['pa'].isel(sl).values; Q = _dh['hus'].isel(sl).values
    Tv = T*(1.0+Q/EPSILON)/(1.0+Q)
    rho0_full += (P/(Rd*Tv)).mean(axis=(0,2,3))*(t1-t0); _nr += (t1-t0)
    del T,P,Q,Tv; gc.collect()
_da.close(); _dp.close(); _dh.close(); del _da,_dp,_dh; gc.collect()
rho0_full /= _nr

def coupe_instant(it):
    """Coupe (z,x) moyennée sur y à l'instant it — TOUS les niveaux z."""
    U_i = np.zeros((_nz_full, n_x)); W_i = np.zeros((_nz_full, n_x)); UP_i = np.zeros((_nz_full, n_x))
    du = xr.open_dataset(path3d('ua')); dw = xr.open_dataset(path3d('wa'))
    for iz in range(_nz_full):
        u_yx = du['ua'].isel({dim_t: it, dim_z: iz}).values
        w_yx = dw['wa'].isel({dim_t: it, dim_z: iz}).values
        u_p = u_yx - u_yx.mean(); w_p = w_yx - w_yx.mean()
        U_i[iz] = u_yx.mean(axis=0); W_i[iz] = w_yx.mean(axis=0)
        UP_i[iz] = (u_p*w_p).mean(axis=0)
        del u_yx, w_yx, u_p, w_p
    du.close(); dw.close(); del du, dw; gc.collect()
    return alt, rho0_full, U_i, W_i, UP_i

def psi_poisson_periodic(U_i, W_i, x, z_i, rho_i):
    """psi de masse : PÉRIODIQUE en x, DIRICHLET INHOMOGÈNE en z.
       bas  : psi(z=0)    = 0
       haut : psi(z=ztop) = ∫_0^ztop rho0(z) <u>(z) dz   (transport de masse net)
    On résout le Poisson sur les niveaux INTÉRIEURS ; les deux niveaux de bord
    (0 et nz-1) sont imposés et passent au second membre.
    """
    nz, nx = U_i.shape
    dxl = float(x[1]-x[0]); dzl = float(z_i[1]-z_i[0])
    rhoU = rho_i[:, None]*U_i
    rhoW = rho_i[:, None]*W_i

    # --- terme source : rotationnel du flux de masse ---
    #     dérivée en x PÉRIODIQUE (np.roll fait le wrap)
    ddx_rhoW = (np.roll(rhoW, -1, axis=1) - np.roll(rhoW, 1, axis=1)) / (2*dxl)
    ddz_rhoU = np.gradient(rhoU, dzl, axis=0)
    omega = ddx_rhoW - ddz_rhoU

    # --- valeurs de Dirichlet aux bords haut/bas (constantes en x) ---
    rho0_ubar = (rho_i * U_i.mean(axis=1))          # rho0(z) * <u>_x(z)  (nz,)
    psi_bot = 0.0
    # ∫_0^ztop rho0 <u> dz  (trapèze maison -> robuste NumPy 1.x/2.x)
    psi_top = float(np.sum(0.5*(rho0_ubar[1:]+rho0_ubar[:-1])*np.diff(z_i)))
    print(f"    psi_bot = {psi_bot:+.3e}   psi_top = {psi_top:+.3e}  (kg m^-1 s^-1)")

    # --- Lx périodique : toutes les nx colonnes sont inconnues ---
    Lx = diags([1., -2., 1.], [-1, 0, 1], shape=(nx, nx), format='lil')
    Lx[0, nx-1] = 1.0       # le point 0 voit le point nx-1 (wrap gauche)
    Lx[nx-1, 0] = 1.0       # le point nx-1 voit le point 0 (wrap droit)
    Lx = Lx.tocsr() / dxl**2

    # --- Lz Dirichlet : seulement les nz-2 niveaux intérieurs ---
    Lz = diags([1., -2., 1.], [-1, 0, 1], shape=(nz-2, nz-2)).tocsr() / dzl**2

    Ix = identity(nx)
    Iz = identity(nz-2)
    A = (kron(Iz, Lx) + kron(Lz, Ix)).tocsr()

    # --- second membre : source intérieure + report des bords Dirichlet ---
    b = omega[1:-1, :].copy()                # (nz-2, nx)
    # le niveau intérieur le plus bas (j=0) a un voisin imposé psi_bot
    b[0,  :] -= psi_bot / dzl**2
    # le niveau intérieur le plus haut (j=nz-3) a un voisin imposé psi_top
    b[-1, :] -= psi_top / dzl**2
    b = b.ravel()

    psi = np.zeros((nz, nx))
    psi[0,  :] = psi_bot
    psi[-1, :] = psi_top
    psi[1:-1, :] = spsolve(A, b).reshape(nz-2, nx)
    return psi

for it in T_SNAPS:
    z_i, rho_i, U_i, W_i, UP_i = coupe_instant(it)
    psi_i = psi_poisson_periodic(U_i, W_i, x, z_i, rho_i)
    Xkm, Zkm = x/1000, z_i/1000

    fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    ax = axes[0]
    wmax = np.percentile(np.abs(W_i), 99) + 1e-12
    pc = ax.pcolormesh(Xkm, Zkm, W_i, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-wmax, vmax=wmax), cmap='RdBu_r')
    ax.contour(Xkm, Zkm, psi_i, levels=14, colors='k', linewidths=0.6)
    ax.set_title(f"(a) Lignes de courant (psi) + <w>_y  -  it = {it}")
    ax.set_ylabel('z [km]'); fig.colorbar(pc, ax=ax, label='<w>_y [m/s]', pad=0.01)

    ax = axes[1]
    fmax = np.percentile(np.abs(UP_i), 98) + 1e-12
    pc = ax.pcolormesh(Xkm, Zkm, UP_i, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-fmax, vmax=fmax), cmap='RdBu_r')
    ax.contour(Xkm, Zkm, psi_i, levels=14, colors='k', linewidths=0.6)
    ax.set_title("(b) Memes lignes de courant + <u'w'>_y")
    ax.set_xlabel('x [km]'); ax.set_ylabel('z [km]')
    fig.colorbar(pc, ax=ax, label="<u'w'>_y [m2/s2]", pad=0.01)
    plt.tight_layout(); plt.show()


In [ ]:
for it in T_SNAPS:
    z_i, rho_i, U_i, W_i, UP_i = coupe_instant(it)
    psi_i = psi_poisson_periodic(U_i, W_i, x, z_i, rho_i)
    Xkm, Zkm = x/1000, z_i/1000

    fig2, ax2 = plt.subplots(figsize=(11, 4))
    pmax = np.percentile(np.abs(psi_i), 99) + 1e-12
    pc2 = ax2.pcolormesh(Xkm, Zkm, psi_i, shading='auto',
                         norm=TwoSlopeNorm(vcenter=0, vmin=-pmax, vmax=pmax),
                         cmap='RdBu_r')
    ax2.set_xlabel('x [km]'); ax2.set_ylabel('z [km]')
    ax2.set_title(f"$\\psi$ (fonction de courant de masse) - it = {it}")
    fig2.colorbar(pc2, ax=ax2, label=r'$\psi$ [kg m$^{-1}$ s$^{-1}$]', pad=0.01)
    plt.tight_layout(); plt.show()


In [ ]:
# ======================================================================
#  Décomposition par zones up/dn/env EN 2D (moyenné sur y)
#  À un instant it : pour chaque z, on moyenne sur y -> profils <u>_y(x), <w>_y(x)
#  Puis on classe les COLONNES x en up/dn/env selon <w>_y(x), seuil w_s = C*std_x(<w>_y)
#
#    Phi2D(z) = somme_k [ T1_k(z) + T2_k(z) ]                 (identité exacte)
#    T1_k(z)  = sigma_k * (uc_k - <u>)(wc_k - <w>)            <- organisé en x
#    T2_k(z)  = sigma_k * < (<u>_y - uc_k)(<w>_y - wc_k) >_k  <- résidu en x intra-cat
#  où moyennes/std sont prises SUR LES COLONNES x du profil moyenné-y.
# ======================================================================
IT_DECOMP = n_t - 1
C_DECOMP  = 0.6
W0_MIN    = 0.001
USE_REG   = False        # True -> ne moyenne que sur les colonnes humides

# >>> tous les niveaux z (aucune troncature verticale) <<<
iz_lst = np.arange(alt.size)
z_D    = alt.copy(); nz_D = z_D.size
cats   = ('up', 'dn', 'env')

phi2d = np.zeros(nz_D)
T1    = {k: np.zeros(nz_D) for k in cats}
T2    = {k: np.zeros(nz_D) for k in cats}
sig   = {k: np.zeros(nz_D) for k in cats}

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values   # (ny, nx)
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values

    # --- moyenne sur y -> profils en x (le "nuage" devient les colonnes x) ---
    if USE_REG:
        uy = u_yx[mh].mean(axis=0) if mh.ndim==1 else np.where(mh, u_yx, np.nan)
        # cas simple : mh est un masque (ny,nx) -> moyenne y pondérée par mh
        uy = np.nanmean(np.where(mh, u_yx, np.nan), axis=0)
        wy = np.nanmean(np.where(mh, w_yx, np.nan), axis=0)
    else:
        uy = u_yx.mean(axis=0)        # <u>_y(x)  (nx,)
        wy = w_yx.mean(axis=0)        # <w>_y(x)

    ubar, wbar, Nx = uy.mean(), wy.mean(), uy.size

    # flux Reynolds "2D" : covariance en x des profils moyennés-y
    phi2d[j] = ((uy - ubar) * (wy - wbar)).mean()

    # seuil sur la dispersion EN x de <w>_y
    ws = max(W0_MIN, C_DECOMP * wy.std())
    masks = {'up': wy > ws, 'dn': wy < -ws, 'env': np.abs(wy) <= ws}

    for k in cats:
        m = masks[k]; Nk = m.sum()
        if Nk == 0:
            continue
        sk = Nk / Nx
        uc = uy[m].mean(); wc = wy[m].mean()
        sig[k][j] = sk
        T1[k][j] = sk * (uc - ubar) * (wc - wbar)                       # organisé
        T2[k][j] = sk * ((uy[m] - uc) * (wy[m] - wc)).mean()            # résidu en x

    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

# --- totaux + contrôle d'identité ---
T1tot = sum(T1[k] for k in cats)
T2tot = sum(T2[k] for k in cats)
err = np.abs(phi2d - (T1tot + T2tot)).max()
print(f"contrôle identité  max|Φ2D − Σ_k(T1_k+T2_k)| = {err:.2e}  (doit être ~1e-18)")

frac_T1 = np.median(np.abs(T1tot) / (np.abs(phi2d) + 1e-30))
frac_T2 = np.median(np.abs(T2tot) / (np.abs(phi2d) + 1e-30))
print(f"poids médian  T1 (organisé x) = {frac_T1:.0%}   T2 (résidu x) = {frac_T2:.0%}")

# ======================================================================
#  FIGURE
# ======================================================================
Zkm = z_D / 1000; sc = 1e3
col = {'up': 'crimson', 'dn': 'royalblue', 'env': 'green'}
fig, ax = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

ax[0].plot(phi2d*sc, Zkm, 'k',   lw=2.5, label=r"$\Phi_{2D}$ (moyenné-y)")
ax[0].plot(T1tot*sc, Zkm, 'r--', lw=2,   label=r'$T_1$ organisé en x')
ax[0].plot(T2tot*sc, Zkm, 'b-.', lw=2,   label=r'$T_2$ résidu en x')
ax[0].axvline(0, color='grey', alpha=.4)
ax[0].set_xlabel(r'flux ($\times10^{-3}$ m²/s²)'); ax[0].set_ylabel('z [km]')
ax[0].set_title('Total 2D = organisé + résidu'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

for k in cats:
    ax[1].plot((T1[k] + T2[k])*sc, Zkm, color=col[k], lw=2, label=k)
ax[1].axvline(0, color='grey', alpha=.4)
ax[1].set_xlabel(r'flux ($\times10^{-3}$)'); ax[1].set_title('Contribution par catégorie')
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)

for k in cats:
    ax[2].plot(sig[k], Zkm, color=col[k], lw=2, label=fr'$\sigma_{{{k}}}$')
ax[2].set_xlabel('fraction des colonnes x'); ax[2].set_title('Aires des catégories (en x)')
ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# ======================================================================
#  Balayage de C : on cherche le C dont T1_tot (rouge) colle le mieux à Phi2D (noir)
#  Phi2D et T1_tot sont tous deux des covariances (m²/s²) -> comparaison directe.
# ======================================================================
IT_DECOMP = n_t - 1
W0_MIN    = 0.001
Cs = np.array([0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0,
               1.2, 1.5, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0, 7.0, 10.0])

# >>> tous les niveaux z (aucune troncature verticale) <<<
iz_lst = np.arange(alt.size)
z_D    = alt.copy(); nz_D = z_D.size
cats   = ('up', 'dn', 'env')

phi2d  = np.zeros(nz_D)
T1tot  = {C: np.zeros(nz_D) for C in Cs}      # le "rouge" pour chaque C

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    uy = u_yx.mean(axis=0); wy = w_yx.mean(axis=0)
    ubar, wbar, Nx = uy.mean(), wy.mean(), uy.size

    phi2d[j] = ((uy - ubar) * (wy - wbar)).mean()     # noir

    sw = wy.std()
    for C in Cs:
        ws = max(W0_MIN, C * sw)
        masks = {'up': wy > ws, 'dn': wy < -ws, 'env': np.abs(wy) <= ws}
        acc = 0.0
        for m in masks.values():
            Nk = m.sum()
            if Nk == 0: continue
            sk = Nk / Nx
            uc = uy[m].mean(); wc = wy[m].mean()
            acc += sk * (uc - ubar) * (wc - wbar)     # T1 par catégorie
        T1tot[C][j] = acc
    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

# --- NSE de chaque C : T1tot vs phi2d ---
def nse_loc(truth, pred):
    ok = np.isfinite(truth) & np.isfinite(pred)
    return 1 - np.nansum((truth[ok]-pred[ok])**2) / np.nansum((truth[ok]-np.nanmean(truth[ok]))**2)

scores = {C: nse_loc(phi2d, T1tot[C]) for C in Cs}
C_star = max(scores, key=scores.get)
print(f"{'C':>6} | {'NSE(T1 vs Phi2D)':>18}")
print('-'*28)
for C in Cs: print(f"{C:6.2f} | {scores[C]:+18.3f}")
print(f"\n>> C* = {C_star}   (NSE = {scores[C_star]:+.3f})")


In [ ]:
# ======================================================================
#  Où vit la covariance within T2 ?  Décomposition Fourier + ondelettes
#  Part du T2[CAT] calculé dans le bloc up/dn/env (même bande, même instant).
#    T2_k/.. = <u'' w''>_x = somme_λ Co(λ)   (Parseval, contrôlé)
#  (a) co-spectre Fourier  Co(λ,z)   -> à quelle échelle
#  (b) co-scalogramme Morlet (x,λ)   -> à quelle échelle ET où (niveau |T2| max)
#  (c) profil T2(z) + échelle dominante par niveau  -> lien spectre <-> physique
# ======================================================================
CAT = 'up'                         # catégorie dont on spectralise le T2

nx  = x.size
dx  = float(x[1] - x[0])           # pas horizontal [m]
print(f"nx={nx}  dx={dx/1000:.2f} km  domaine={nx*dx/1000:.0f} km")

# --- garde-fou : T2 doit être sur la même bande que ce bloc ---
assert T2[CAT].shape[0] == nz_D, (
    f"T2[{CAT}] a {T2[CAT].shape[0]} niveaux, bande={nz_D} -> "
    "relance le bloc up/dn/env avec la MÊME bande avant celui-ci.")

# --- axes spectraux Fourier ---
kx     = np.fft.rfftfreq(nx, d=dx) * 2*np.pi          # nombre d'onde [rad/m]
lam_km = (2*np.pi / np.maximum(kx, 1e-12)) / 1000     # longueur d'onde [km]

# --- conteneurs ---
Co_T2   = np.zeros((nz_D, kx.size))   # co-spectre par niveau
T2_chk  = np.zeros(nz_D)              # T2 reconstruit (contrôle)
upp_all = np.zeros((nz_D, nx))        # u''(x) (0 hors catégorie) pour ondelettes
wpp_all = np.zeros((nz_D, nx))

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    uy = u_yx.mean(axis=0); wy = w_yx.mean(axis=0)      # <u>_y(x), <w>_y(x)

    ws = max(W0_MIN, C_DECOMP * wy.std())
    masks = {'up': wy > ws, 'dn': wy < -ws, 'env': np.abs(wy) <= ws}
    m = masks[CAT]

    # anomalies intra-catégorie, définies sur TOUT x (0 hors cat)
    upp = np.zeros(nx); wpp = np.zeros(nx)
    if m.any():
        upp[m] = uy[m] - uy[m].mean()
        wpp[m] = wy[m] - wy[m].mean()
    upp_all[j] = upp; wpp_all[j] = wpp

    T2_chk[j] = (upp * wpp).mean()        # == T2[CAT][j]

    Fu = np.fft.rfft(upp); Fw = np.fft.rfft(wpp)
    co = np.real(Fu * np.conj(Fw)) / nx**2
    co[1:] *= 2                            # repli fréquences négatives
    Co_T2[j] = co
    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

# --- contrôles ---
err_p = np.abs(Co_T2.sum(axis=1) - T2_chk).max()
err_c = np.abs(T2_chk - T2[CAT]).max()
print(f"[Parseval]  max|sum_λ Co - T2|       = {err_p:.2e}  (~1e-18)")
print(f"[cohérence] max|T2_spectre - T2[{CAT}]| = {err_c:.2e}  (~1e-18)")

k_max = int(np.argmax(np.abs(T2_chk)))
print(f"niveau |T2_{CAT}| max : z = {z_D[k_max]/1000:.1f} km   T2 = {T2_chk[k_max]:.2e}")

# --- échelle dominante par niveau (barycentre |Co| en log-λ, hors k=0) ---
lam_dom = np.zeros(nz_D)
for j in range(nz_D):
    w_ = np.abs(Co_T2[j, 1:])
    lam_dom[j] = np.exp(np.sum(w_*np.log(lam_km[1:])) / (w_.sum()+1e-30)) if w_.sum()>0 else np.nan


# ======================================================================
#  FIGURE  (a) co-spectre  
# ======================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))

A  = Co_T2[:, 1:]                              # <-- plus de kx multiplié
mA = np.percentile(np.abs(A), 99) + 1e-30
pc0 = ax.pcolormesh(lam_km[1:], z_D/1000, A, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-mA, vmax=mA), cmap='RdBu_r')
ax.plot(lam_dom, z_D/1000, 'k-', lw=1.5, alpha=.6, label='échelle dominante')
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel(r'$\lambda$ [km]'); ax.set_ylabel('z [km]')
ax.set_title(fr'(a) Co-spectre brut de $T_2^{{{CAT}}}$')
ax.legend(fontsize=8, loc='upper left')
'''fig.colorbar(pc0, ax=ax[0], label=r'$\mathrm{Co}$', pad=0.01)   # label sans le k'''

plt.tight_layout(); plt.show()

## Visualisation de  w̄_up / w̄_dn / w̄_env (après moyenne-y) et leur rôle dans T1/T2

In [ ]:
# ======================================================================
#  ÉTAPE 3 — w̄_up / w̄_dn / w̄_env (après moyenne-y) et leur rôle dans T1/T2
#  ----------------------------------------------------------------------
#  À it = IT_DECOMP, pour chaque z :
#    profils moyennés-y  uy(x), wy(x)  ->  catégories up/dn/env (seuil w_s)
#    w̄_k(z) = moyenne de wy sur les colonnes de la catégorie k
#    u_k(z) = idem pour u   ;   σ_k(z) = fraction de colonnes de k
#    T1_k(z)= σ_k (u_k-ū)(w̄_k-w̄)              (organisé)
#    T2_k(z)= σ_k ⟨(uy-u_k)(wy-w̄_k)⟩_{x∈k}     (résidu intra-catégorie)
#  -> on visualise w̄_k(z) et on les confronte à T1_k, T2_k.
#  Mêmes conventions que le bloc up/dn/env (IT_DECOMP, C_DECOMP, W0_MIN).
# ======================================================================
CATS = ('up', 'dn', 'env')

wbar_k = {k: np.zeros(nz_D) for k in CATS}     # w̄_k(z)
ubar_k = {k: np.zeros(nz_D) for k in CATS}     # u_k(z)
sig_k  = {k: np.zeros(nz_D) for k in CATS}     # σ_k(z)
T1_k   = {k: np.zeros(nz_D) for k in CATS}
T2_k   = {k: np.zeros(nz_D) for k in CATS}
wbar_glob = np.zeros(nz_D)                     # w̄ global (référence)

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    uy = u_yx.mean(axis=0); wy = w_yx.mean(axis=0)
    ubar, wbar, Nx = uy.mean(), wy.mean(), uy.size
    wbar_glob[j] = wbar

    ws = max(W0_MIN, C_DECOMP * wy.std())
    masks = {'up': wy > ws, 'dn': wy < -ws, 'env': np.abs(wy) <= ws}

    for k in CATS:
        m = masks[k]; Nk = m.sum()
        if Nk == 0:
            wbar_k[k][j] = np.nan; ubar_k[k][j] = np.nan
            continue
        sk = Nk / Nx
        uc = uy[m].mean(); wc = wy[m].mean()
        sig_k[k][j]  = sk
        ubar_k[k][j] = uc
        wbar_k[k][j] = wc                                   # <-- w̄_k(z)
        T1_k[k][j]   = sk * (uc - ubar) * (wc - wbar)
        T2_k[k][j]   = sk * ((uy[m] - uc) * (wy[m] - wc)).mean()

    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

# --- bilan de masse conditionnel : Σ_k σ_k w̄_k doit ≈ w̄ global ---
mass_resid = np.nanmax(np.abs(
    sum(sig_k[k]*np.nan_to_num(wbar_k[k]) for k in CATS) - wbar_glob))
print(f"[contrôle masse] max|Σ σ_k w̄_k − w̄| = {mass_resid:.2e}")

# ======================================================================
#  FIGURE — w̄_k(z)  |  T1_k(z)  |  T2_k(z)
# ======================================================================
col = {'up': 'crimson', 'dn': 'royalblue', 'env': 'seagreen'}
Zkm = z_D / 1000; sc = 1e3

fig, ax = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

# (a) vitesses verticales conditionnelles
for k in CATS:
    ax[0].plot(wbar_k[k], Zkm, color=col[k], lw=2, label=fr'$\bar w_{{{k}}}$')
ax[0].plot(wbar_glob, Zkm, 'k--', lw=1.4, alpha=.7, label=r'$\bar w$ global')
ax[0].axvline(0, color='grey', alpha=.4)
ax[0].set_xlabel(r'$\bar w_k$ [m/s]'); ax[0].set_ylabel('z [km]')
ax[0].set_title('(a) Vitesse verticale conditionnelle'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

# (b) rôle dans T1 (organisé)
for k in CATS:
    ax[1].plot(T1_k[k]*sc, Zkm, color=col[k], lw=2, label=fr'$T_1^{{{k}}}$')
ax[1].plot(sum(T1_k[k] for k in CATS)*sc, Zkm, 'k', lw=2.2, label=r'$T_1$ total')
ax[1].axvline(0, color='grey', alpha=.4)
ax[1].set_xlabel(r'$T_1$ ($\times10^{-3}$ m²/s²)')
ax[1].set_title('(b) Rôle dans $T_1$ (organisé)'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)

# (c) rôle dans T2 (résidu)
for k in CATS:
    ax[2].plot(T2_k[k]*sc, Zkm, color=col[k], lw=2, label=fr'$T_2^{{{k}}}$')
ax[2].plot(sum(T2_k[k] for k in CATS)*sc, Zkm, 'k', lw=2.2, label=r'$T_2$ total')
ax[2].axvline(0, color='grey', alpha=.4)
ax[2].set_xlabel(r'$T_2$ ($\times10^{-3}$ m²/s²)')
ax[2].set_title('(c) Rôle dans $T_2$ (résidu)'); ax[2].legend(fontsize=9); ax[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

# --- petit tableau récap au niveau du T1 max ---
jm = int(np.nanargmax(np.abs(sum(T1_k[k] for k in CATS))))
print(f"\nAu niveau T1 max (z = {Zkm[jm]:.1f} km) :")
print(f"{'cat':>5} | {'σ_k':>6} | {'w̄_k [m/s]':>10} | {'T1_k':>10} | {'T2_k':>10}")
print('-'*52)
for k in CATS:
    print(f"{k:>5} | {sig_k[k][jm]:6.3f} | {wbar_k[k][jm]:10.4f} | "
          f"{T1_k[k][jm]:10.2e} | {T2_k[k][jm]:10.2e}")

## Analyse spectrale

In [ ]:
# ======================================================================
#  ÉTAPE 1 — Co-spectre Fourier de la covariance ORGANISÉE T1 par niveau
#  ----------------------------------------------------------------------
#  Idée : T1(z) = covariance en x des profils "par paliers" reconstruits
#         u_org(x) = uc[cat(x)] - ubar,  w_org(x) = wc[cat(x)] - wbar
#  où cat(x) ∈ {up,dn,env} est la catégorie de la colonne x (seuil sur w̄_y).
#  Le co-spectre de (u_org, w_org) vérifie  Σ_λ Co1(λ) = T1(z)  (Parseval),
#  et révèle À QUELLE ÉCHELLE horizontale vit le flux de masse organisé.
#
#  Dépend de : x, alt, iz_lst, z_D, nz_D, dim_t, dim_z, path3d,
#              IT_DECOMP, C_DECOMP, W0_MIN  (mêmes conventions que up/dn/env).
# ======================================================================
CAT_LIST = ('up', 'dn', 'env')

nx  = x.size
dx  = float(x[1] - x[0])
print(f"[T1 spectre] nx={nx}  dx={dx/1000:.2f} km  domaine={nx*dx/1000:.0f} km")

# --- axes spectraux Fourier (rfft : fréquences positives) ---
kx     = np.fft.rfftfreq(nx, d=dx) * 2*np.pi          # nombre d'onde [rad/m]
lam_km = (2*np.pi / np.maximum(kx, 1e-12)) / 1000     # longueur d'onde [km]

# --- conteneurs ---
Co_T1     = np.zeros((nz_D, kx.size))   # co-spectre du flux organisé, par niveau
T1_chk    = np.zeros(nz_D)              # T1 reconstruit (contrôle Parseval)
T1_ref    = np.zeros(nz_D)              # T1 "direct" (somme des catégories) pour cohérence
uorg_all  = np.zeros((nz_D, nx))        # u_org(x) par niveau (utile plus bas)
worg_all  = np.zeros((nz_D, nx))

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values   # (ny, nx)
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values

    # --- moyenne sur y -> profils en x ---
    uy = u_yx.mean(axis=0); wy = w_yx.mean(axis=0)
    ubar, wbar = uy.mean(), wy.mean()

    # --- classification des colonnes x (même seuil que le bloc up/dn/env) ---
    ws = max(W0_MIN, C_DECOMP * wy.std())
    masks = {'up': wy > ws, 'dn': wy < -ws, 'env': np.abs(wy) <= ws}

    # --- reconstruction par paliers : valeur de catégorie - moyenne globale ---
    u_org = np.zeros(nx); w_org = np.zeros(nx)
    t1_direct = 0.0
    for k in CAT_LIST:
        m = masks[k]; Nk = m.sum()
        if Nk == 0:
            continue
        uc = uy[m].mean(); wc = wy[m].mean()
        u_org[m] = uc - ubar
        w_org[m] = wc - wbar
        t1_direct += (Nk / nx) * (uc - ubar) * (wc - wbar)   # = σ_k (uc-ū)(wc-w̄)

    uorg_all[j] = u_org; worg_all[j] = w_org
    T1_ref[j]   = t1_direct
    T1_chk[j]   = (u_org * w_org).mean()        # doit égaler t1_direct (paliers constants)

    # --- co-spectre Fourier de (u_org, w_org), normalisé Parseval ---
    Fu = np.fft.rfft(u_org); Fw = np.fft.rfft(w_org)
    co = np.real(Fu * np.conj(Fw)) / nx**2
    if nx % 2 == 0:
        co[1:-1] *= 2          # repli des fréquences négatives (sauf 0 et Nyquist)
    else:
        co[1:]   *= 2
    Co_T1[j] = co

    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

# ----------------------------------------------------------------------
#  Contrôles
# ----------------------------------------------------------------------
err_parseval = np.abs(Co_T1.sum(axis=1) - T1_chk).max()
err_direct   = np.abs(T1_chk - T1_ref).max()
print(f"[Parseval]  max|Σ_λ Co1 - T1_chk|   = {err_parseval:.2e}   (≈ 1e-18)")
print(f"[cohérence] max|T1_chk - T1_direct| = {err_direct:.2e}     (≈ 1e-18)")

j_max = int(np.argmax(np.abs(T1_chk)))
print(f"niveau |T1| max : z = {z_D[j_max]/1000:.1f} km   (T1 = {T1_chk[j_max]:.2e} m²/s²)")

# --- échelle dominante par niveau (barycentre de |Co1| en log-λ) ---
lam_dom_T1 = np.full(nz_D, np.nan)
for j in range(nz_D):
    wgt = np.abs(Co_T1[j, 1:])
    if wgt.sum() > 0:
        lam_dom_T1[j] = np.exp(np.sum(wgt * np.log(lam_km[1:])) / wgt.sum())

# ======================================================================
#  FIGURE — co-spectre de T1 (area-preserving k·Co) + échelle dominante
# ======================================================================
A1 = kx[None, 1:] * Co_T1[:, 1:]          # k·Co : contribution par décade log

# borne de couleur robuste (symétrique autour de 0) ; garde-fou si A1 ≡ 0
m1 = np.percentile(np.abs(A1), 99)
if not np.isfinite(m1) or m1 <= 0:
    m1 = np.nanmax(np.abs(A1))
if not np.isfinite(m1) or m1 <= 0:
    m1 = 1e-30

fig, ax = plt.subplots(figsize=(8, 6))

pc = ax.pcolormesh(lam_km[1:], z_D/1000, A1, shading='auto',
                   norm=TwoSlopeNorm(vcenter=0, vmin=-m1, vmax=m1), cmap='RdBu_r')
ax.plot(lam_dom_T1, z_D/1000, 'k-', lw=1.6, alpha=.7, label='échelle dominante')
ax.set_xscale('log'); ax.invert_xaxis()
ax.set_xlabel(r'$\lambda$ [km]'); ax.set_ylabel('z [km]')
ax.set_title(r'Co-spectre du flux organisé $T_1$  ($k\,\mathrm{Co}$)')
ax.legend(fontsize=9, loc='upper left')

# colorbar attachée explicitement au mappable et à l'axe -> s'affiche correctement
cbar = fig.colorbar(pc, ax=ax, label=r'$k\,\mathrm{Co}_{T_1}$', pad=0.02)

plt.tight_layout(); plt.show()




In [ ]:
# ======================================================================
#  CO-SCALOGRAMME 1D-EN-X PAR NIVEAU — préserve l'info en x
#  ----------------------------------------------------------------------
#  CWT Morlet sur x (à z fixé) de u_org et w_org :
#      Wu(b,s) = (u_org ⋆ ψ*_{s,b}),   Ww(b,s) = (w_org ⋆ ψ*_{s,b})
#  Co-scalogramme local :  Co(b,s) = Re[ Wu(b,s) · conj(Ww(b,s)) ]
#  -> carte (x, λ) : OÙ en x vit le flux organisé ET à quelle échelle.
#
#  Reconstruction admissible (Torrence & Compo 1998) :
#      T1(z) ≈ (δj·δt / C_δ) · Σ_s Co(b,s)/s      moyenné en b
#  -> on vérifie que le co-scalogramme intègre bien vers T1 (à C_δ près).
#
#  x périodique -> CWT par FFT exacte (pas de bord en x).
#  Réutilise : uorg_all, worg_all (de l'ÉTAPE 1), x, z_D, nz_D, nx, dx.
# ======================================================================
W0_MORLET = 6.0                                  # ω0 (Morlet standard)
NSCALE    = 64

dx = float(x[1] - x[0])
# --- échelles : de ~2 dx (Nyquist) à ~domaine/2, espacées en log ---
scales = np.geomspace(2*dx, nx*dx/2, NSCALE)     # [m]
# facteur échelle s -> longueur d'onde de Fourier λ (Torrence & Compo)
fourier_factor = (4*np.pi) / (W0_MORLET + np.sqrt(2 + W0_MORLET**2))
lam_w_km = scales * fourier_factor / 1000.0      # [km], 1 par échelle

# constante d'admissibilité de la Morlet (pour la reconstruction)
C_DELTA = 0.776                                  # valeur tabulée pour ω0=6
dj = np.log2(scales[1]/scales[0])                # pas log2 entre échelles

# fréquences angulaires du domaine périodique
k_full = np.fft.fftfreq(nx, d=dx) * 2*np.pi      # [rad/m]

def cwt_morlet_x(sig):
    """CWT Morlet de `sig` (périodique en x) via FFT. Retourne W[NSCALE, nx] complexe."""
    Sig = np.fft.fft(sig)
    out = np.empty((NSCALE, nx), dtype=complex)
    for i, s in enumerate(scales):
        sk   = s * k_full
        # Morlet dans Fourier : normalisée énergie, partie analytique (sk>0)
        psih = (np.pi**-0.25) * np.sqrt(2*np.pi*s/dx) * (sk > 0) * np.exp(-0.5*(sk - W0_MORLET)**2)
        out[i] = np.fft.ifft(Sig * psih)
    return out

def coscalogram_level(j):
    """Co-scalogramme Co(λ, x) au niveau j (indice dans la bande z_D)."""
    Wu = cwt_morlet_x(uorg_all[j])
    Ww = cwt_morlet_x(worg_all[j])
    return np.real(Wu * np.conj(Ww))             # (NSCALE, nx)

# ======================================================================
#  CO-SCALOGRAMME (x, λ) — VERSION LISIBLE
#  Réutilise : cwt_morlet_x, coscalogram_level (ou coscalogram_T1),
#              uorg_all, worg_all, x, z_D, lam_w_km, scales
# ======================================================================
Z_PICK_KM = 5.0
j_pick = int(np.argmin(np.abs(z_D/1000 - Z_PICK_KM)))
Co_xl  = coscalogram_level(j_pick)               # (NSCALE, nx)

xkm = x / 1000.0
T1_lvl = (uorg_all[j_pick] * worg_all[j_pick]).mean()

# échelle dominante (barycentre en log-λ de la marginale |Co|)
marg = np.abs(Co_xl).mean(axis=1)
lam_dom = np.exp(np.sum(marg*np.log(lam_w_km)) / (marg.sum()+1e-30))

fig = plt.figure(figsize=(13, 7))
gs  = fig.add_gridspec(2, 2, width_ratios=[4, 1.1], height_ratios=[1, 3],
                       hspace=0.08, wspace=0.06)

# ----------------------------------------------------------------------
#  (haut) profils organisés u_org et w_org, sur deux axes y distincts
# ----------------------------------------------------------------------
axu = fig.add_subplot(gs[0, 0])
axu.plot(xkm, uorg_all[j_pick], color='navy', lw=1.0, label=r'$u_{org}(x)$')
axu.set_ylabel(r'$u_{org}$  [m/s]', color='navy', fontsize=9)
axu.tick_params(axis='y', labelcolor='navy', labelsize=8)
axu.set_xticklabels([]); axu.margins(x=0)
axu.axhline(0, color='grey', lw=.6, alpha=.5)

axw = axu.twinx()                                # 2e axe y pour w_org
axw.plot(xkm, worg_all[j_pick], color='firebrick', lw=1.0, alpha=.8, label=r'$w_{org}(x)$')
axw.set_ylabel(r'$w_{org}$  [m/s]', color='firebrick', fontsize=9)
axw.tick_params(axis='y', labelcolor='firebrick', labelsize=8)
axw.margins(x=0)
axu.set_title(f"Co-scalogramme du flux organisé $T_1$  —  z = {z_D[j_pick]/1000:.1f} km   "
              f"($T_1$ = {T1_lvl:.2e} m²/s²)", fontsize=11)

# ----------------------------------------------------------------------
#  (centre) carte (x, λ) — le cœur
# ----------------------------------------------------------------------
axm = fig.add_subplot(gs[1, 0], sharex=axu)
mC = np.percentile(np.abs(Co_xl), 99) + 1e-30
pc = axm.pcolormesh(xkm, lam_w_km, Co_xl, shading='auto',
                    norm=TwoSlopeNorm(vcenter=0, vmin=-mC, vmax=mC), cmap='RdBu_r')
axm.set_yscale('log'); axm.invert_yaxis()        # grandes échelles en bas
# graduations λ explicites en km
lam_ticks = [2, 5, 10, 20, 50, 100, 200]
lam_ticks = [t for t in lam_ticks if lam_w_km.min() <= t <= lam_w_km.max()]
axm.set_yticks(lam_ticks); axm.set_yticklabels([str(t) for t in lam_ticks])
axm.axhline(lam_dom, color='k', lw=1.2, ls='--', alpha=.7)
axm.text(xkm[2], lam_dom, f'  λ dom ≈ {lam_dom:.0f} km',
         va='bottom', ha='left', fontsize=8, color='k')
axm.set_xlabel('position  x  [km]', fontsize=10)
axm.set_ylabel(r'échelle  $\lambda$  [km]', fontsize=10)
axm.margins(x=0)

# ----------------------------------------------------------------------
#  (droite) marginale : co-spectre moyenné en x (= ce que donnerait Fourier)
# ----------------------------------------------------------------------
axr = fig.add_subplot(gs[1, 1], sharey=axm)
axr.plot(Co_xl.mean(axis=1), lam_w_km, 'k', lw=1.5)
axr.axvline(0, color='grey', alpha=.5)
axr.axhline(lam_dom, color='k', lw=1.0, ls='--', alpha=.6)
axr.set_yscale('log'); axr.invert_yaxis()
axr.set_yticks(lam_ticks); axr.set_yticklabels([])
axr.set_xlabel(r'$\langle Co\rangle_x$', fontsize=9)
axr.tick_params(labelsize=8)
axr.set_title('moyenne en x', fontsize=9)
axr.margins(y=0)

cbar = fig.colorbar(pc, ax=[axm, axr], label=r'co-flux  $\mathrm{Co}(x,\lambda)$  [m²/s²]',
                    pad=0.02, location='bottom', shrink=0.9)
plt.show()

# ----------------------------------------------------------------------
#  légende textuelle imprimée (aide-mémoire de lecture)
# ----------------------------------------------------------------------
print("COMMENT LIRE :")
print(f"  • axe x (horizontal) = position dans le domaine [km]")
print(f"  • axe λ (vertical)   = taille de la structure [km], grandes échelles en bas")
print(f"  • ROUGE  = flux organisé POSITIF localisé là, à cette échelle")
print(f"  • BLEU   = flux organisé NÉGATIF")
print(f"  • BLANC  = pas de structure (plage calme)")
print(f"  • chaque 'bulbe' = une structure cohérente (un updraft/downdraft)")
print(f"  • λ dominante de ce niveau ≈ {lam_dom:.0f} km")

In [ ]:
# ======================================================================
#  CO-SCALOGRAMME 1D-EN-X PAR NIVEAU — variance-preserving + axes gradués
#  ----------------------------------------------------------------------
#  Pondération Co/s : retire le biais d'échelle du scalogramme (les gros λ
#  ne sont plus gonflés artificiellement ; aire en log-λ ∝ flux).
#  x périodique -> CWT par FFT exacte. Réutilise uorg_all, worg_all, x, z_D.
# ======================================================================
W0_MORLET = 6.0
NSCALE    = 64

dx = float(x[1] - x[0]); nx = x.size
scales = np.geomspace(2*dx, nx*dx/2, NSCALE)
fourier_factor = (4*np.pi) / (W0_MORLET + np.sqrt(2 + W0_MORLET**2))
lam_w_km = scales * fourier_factor / 1000.0
C_DELTA = 0.776
dj = np.log2(scales[1]/scales[0])
k_full = np.fft.fftfreq(nx, d=dx) * 2*np.pi

def cwt_morlet_x(sig):
    Sig = np.fft.fft(sig)
    out = np.empty((NSCALE, nx), dtype=complex)
    for i, s in enumerate(scales):
        sk = s * k_full
        psih = (np.pi**-0.25) * np.sqrt(2*np.pi*s/dx) * (sk > 0) * np.exp(-0.5*(sk - W0_MORLET)**2)
        out[i] = np.fft.ifft(Sig * psih)
    return out

def coscalogram_level(j):
    Wu = cwt_morlet_x(uorg_all[j])
    Ww = cwt_morlet_x(worg_all[j])
    return np.real(Wu * np.conj(Ww))

# ----------------------------------------------------------------------
#  Niveau choisi
# ----------------------------------------------------------------------
Z_PICK_KM = 5.0
j_pick = int(np.argmin(np.abs(z_D/1000 - Z_PICK_KM)))
Co_xl  = coscalogram_level(j_pick)               # co-scalo BRUT
Co_vp  = Co_xl / scales[:, None]                 # <-- variance-preserving (Co/s)

xkm = x / 1000.0
Lx  = xkm[-1] + (xkm[1] - xkm[0])
T1_lvl = (uorg_all[j_pick] * worg_all[j_pick]).mean()

# échelle dominante (sur la version variance-preserving)
marg = np.abs(Co_vp).mean(axis=1)
lam_dom = np.exp(np.sum(marg*np.log(lam_w_km)) / (marg.sum()+1e-30))

# ======================================================================
#  FIGURE
# ======================================================================
fig = plt.figure(figsize=(13, 7))
gs  = fig.add_gridspec(2, 2, width_ratios=[4, 1.1], height_ratios=[1, 3],
                       hspace=0.08, wspace=0.06)

# ----------------------------------------------------------------------
#  (haut) profils organisés u_org et w_org, deux axes y distincts
# ----------------------------------------------------------------------
axu = fig.add_subplot(gs[0, 0])
axu.plot(xkm, uorg_all[j_pick], color='navy', lw=1.0, label=r'$u_{org}(x)$')
axu.set_ylabel(r'$u_{org}$  [m/s]', color='navy', fontsize=9)
axu.tick_params(axis='y', labelcolor='navy', labelsize=8)
axu.set_xticklabels([]); axu.margins(x=0)
axu.axhline(0, color='grey', lw=.6, alpha=.5)

axw = axu.twinx()
axw.plot(xkm, worg_all[j_pick], color='firebrick', lw=1.0, alpha=.8, label=r'$w_{org}(x)$')
axw.set_ylabel(r'$w_{org}$  [m/s]', color='firebrick', fontsize=9)
axw.tick_params(axis='y', labelcolor='firebrick', labelsize=8)
axw.margins(x=0)
axu.set_title(f"Co-scalogramme du flux organisé $T_1$ (variance-preserving) — "
              f"z = {z_D[j_pick]/1000:.1f} km   ($T_1$ = {T1_lvl:.2e} m²/s²)", fontsize=11)

# ----------------------------------------------------------------------
#  (centre) carte (x, λ) variance-preserving
# ----------------------------------------------------------------------
axm = fig.add_subplot(gs[1, 0], sharex=axu)
mC = np.percentile(np.abs(Co_vp), 99) + 1e-30
pc = axm.pcolormesh(xkm, lam_w_km, Co_vp, shading='auto',
                    norm=TwoSlopeNorm(vcenter=0, vmin=-mC, vmax=mC), cmap='RdBu_r')
axm.set_yscale('log'); axm.invert_yaxis()        # grandes échelles en bas

# --- axe Y : λ en km, graduations explicites ---
lam_ticks = [t for t in [2, 5, 10, 20, 50, 100, 200] if lam_w_km.min() <= t <= lam_w_km.max()]
axm.set_yticks(lam_ticks)
axm.set_yticklabels([str(t) for t in lam_ticks])
axm.set_ylabel(r'échelle  $\lambda$  [km]', fontsize=10)

# --- axe X : position en km, graduations tous les 50 km + lignes repères ---
xticks = np.arange(0, Lx + 1, 50)
axm.set_xticks(xticks)
axm.set_xticklabels([f'{int(t)}' for t in xticks])
axm.set_xlabel('position  x  [km]', fontsize=10)
for xr_ in xticks[1:-1]:
    axm.axvline(xr_, color='grey', lw=0.4, ls=':', alpha=0.4)
for lam_ref in [10, 50, 100]:
    if lam_w_km.min() <= lam_ref <= lam_w_km.max():
        axm.axhline(lam_ref, color='grey', lw=0.5, ls=':', alpha=0.5)

# échelle dominante en évidence
axm.axhline(lam_dom, color='k', lw=1.3, ls='--', alpha=.8)
axm.text(xkm[2], lam_dom, f'  λ dom ≈ {lam_dom:.0f} km',
         va='bottom', ha='left', fontsize=8, color='k',
         bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7))
axm.margins(x=0)

# ----------------------------------------------------------------------
#  (droite) marginale variance-preserving : ⟨Co/s⟩_x
# ----------------------------------------------------------------------
axr = fig.add_subplot(gs[1, 1], sharey=axm)
axr.plot(Co_vp.mean(axis=1), lam_w_km, 'k', lw=1.5)
axr.axvline(0, color='grey', alpha=.5)
axr.axhline(lam_dom, color='k', lw=1.0, ls='--', alpha=.6)
axr.set_yscale('log'); axr.invert_yaxis()
axr.set_yticks(lam_ticks); axr.set_yticklabels([])
axr.set_xlabel(r'$\langle Co/s\rangle_x$', fontsize=9)
axr.tick_params(labelsize=8)
axr.set_title('moyenne en x', fontsize=9)
axr.margins(y=0)

cbar = fig.colorbar(pc, ax=[axm, axr],
                    label=r'co-flux variance-preserving  $\mathrm{Co}(x,\lambda)/s$',
                    pad=0.02, location='bottom', shrink=0.9)
plt.show()

# ----------------------------------------------------------------------
#  aide-mémoire de lecture
# ----------------------------------------------------------------------
print("COMMENT LIRE :")
print(f"  • axe x = position [km], graduations tous les 50 km (0 → {Lx:.0f} km)")
print(f"  • axe λ = taille de structure [km] ; PETITES échelles en HAUT, GRANDES en BAS")
print(f"  • variance-preserving (Co/s) : aire en log-λ ∝ flux -> gros λ non gonflés")
print(f"  • ROUGE = flux organisé positif localisé ; BLEU = négatif ; BLANC = calme")
print(f"  • tiret noir = échelle dominante ≈ {lam_dom:.0f} km")

---
# 4. Ondelettes 2D $(x,z)$ — localiser les tourbillons *avant* de fixer $z$

Jusqu'ici l'analyse spectrale était **1D-en-$x$ à $z$ fixé** (cellules co-spectre / co-scalogramme).
La feuille de route demande l'étape suivante : **ne plus fixer $z$ a priori** et chercher les
structures cohérentes directement dans le **plan $(x,z)$**, avec une ondelette qui localise
vraiment en 2D.

**Ondelette de Morlet 2D anisotrope — 4 paramètres.** On prend
$$
\psi_{\,b_x,b_z,\,\lambda_x,\lambda_z}(x,z)\;=\;
\underbrace{e^{\,i\,(k_x x + k_z z)}}_{\text{onde porteuse}}\;
\underbrace{e^{-\tfrac12\!\left[(x-b_x)^2/\sigma_x^2+(z-b_z)^2/\sigma_z^2\right]}}_{\text{enveloppe gaussienne}}
$$
soit **4 paramètres** exactement, comme noté à la main :
- $(b_x,b_z)$ : **2 localisations** — *où* est la structure ;
- $(\lambda_x,\lambda_z)$ : **2 longueurs d'onde** — *quelle taille* en $x$ et en $z$,
  avec $k_x=2\pi/\lambda_x$, $k_z=2\pi/\lambda_z$, et largeur d'enveloppe liée à l'échelle
  ($\sigma=\omega_0\lambda/2\pi$, même convention $\omega_0=6$ que le 1D).

**Pourquoi c'est la bonne décompo pour ce problème ?** Un *rouleau* (« mon gros tourbillon »)
est un objet **anisotrope** : étendu en $x$, confiné en $z$. Une base isotrope (un seul $\lambda$)
le rate ; ici on autorise $\lambda_x\neq\lambda_z$ et une **orientation**, ce qui colle à la
géométrie d'un overturning. La transformée garde la **position** (contrairement à Fourier) tout
en séparant les **échelles** (contrairement à un simple filtre).

**Sur quel champ ?** On applique la CWT 2D au flux organisé porté par la circulation. Le marqueur
le plus direct d'un tourbillon dans le plan $(x,z)$ est la **vorticité de la circulation
moyennée-$y$**, $\zeta=\partial_x \bar w-\partial_z \bar u$ (cœur = extremum de $\zeta$), qu'on
relie ensuite à $\psi$ (section 1bis) et au flux $u'w'$.

In [ ]:
# ======================================================================
#  4a — Coupe (z,x) moyennée-y à IT_DECOMP : <u>_y(z,x), <w>_y(z,x)
#       + vorticité de circulation  zeta = d_x <w> - d_z <u>
#       + fonction de courant psi (réutilise psi_poisson_periodic)
#  TOUS les niveaux z conservés. Accès NetCDF niveau-par-niveau (RAM maîtrisée).
#  Dépend de : path3d, dim_t, dim_z, n_x, x, alt, iz_lst, z_D, nz_D,
#              IT_DECOMP, rho0_full, psi_poisson_periodic.
# ======================================================================
import numpy as np, xarray as xr, gc

# --- garde-fou : rho0_full doit exister (calculé dans le bloc lignes de courant) ---
if 'rho0_full' not in dir():
    _nz_full = alt.size
    rho0_full = np.zeros(_nz_full); _nr = 0
    _da = xr.open_dataset(path3d('ta')); _dp = xr.open_dataset(path3d('pa')); _dh = xr.open_dataset(path3d('hus'))
    for t0 in range(t_stat, n_t, BLOC):
        t1 = min(t0+BLOC, n_t); sl = {dim_t: slice(t0,t1)}
        T = _da['ta'].isel(sl).values; P = _dp['pa'].isel(sl).values; Q = _dh['hus'].isel(sl).values
        Tv = T*(1.0+Q/EPSILON)/(1.0+Q)
        rho0_full += (P/(Rd*Tv)).mean(axis=(0,2,3))*(t1-t0); _nr += (t1-t0)
        del T,P,Q,Tv; gc.collect()
    _da.close(); _dp.close(); _dh.close(); del _da,_dp,_dh; gc.collect()
    rho0_full /= _nr

Uy2d = np.zeros((nz_D, n_x))      # <u>_y(z,x)
Wy2d = np.zeros((nz_D, n_x))      # <w>_y(z,x)

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
for j, iz in enumerate(iz_lst):
    u_yx = ds_u['ua'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    w_yx = ds_w['wa'].isel({dim_t: IT_DECOMP, dim_z: iz}).values
    Uy2d[j] = u_yx.mean(axis=0)
    Wy2d[j] = w_yx.mean(axis=0)
    del u_yx, w_yx; gc.collect()
ds_u.close(); ds_w.close(); del ds_u, ds_w; gc.collect()

dxl = float(x[1]-x[0])
dzl = float(np.mean(np.diff(z_D)))

# --- vorticité de circulation : d_x periodique (roll), d_z non-uniforme (gradient) ---
dWdx = (np.roll(Wy2d, -1, axis=1) - np.roll(Wy2d, 1, axis=1)) / (2*dxl)
dUdz = np.gradient(Uy2d, z_D, axis=0)
zeta = dWdx - dUdz                       # (nz_D, n_x)  [s^-1]

# --- psi cohérent avec la section 1bis (mêmes BCs) ---
psi2d = psi_poisson_periodic(Uy2d, Wy2d, x, z_D, rho0_full)

print(f"coupe (z,x) prête : {Uy2d.shape}  dx={dxl/1000:.2f} km  dz≈{dzl:.0f} m")
print(f"zeta : min={zeta.min():+.2e}  max={zeta.max():+.2e} s^-1")
zj, zi = np.unravel_index(np.argmax(np.abs(zeta)), zeta.shape)
print(f"|zeta| max @ z={z_D[zj]/1000:.1f} km, x={x[zi]/1000:.0f} km")

In [ ]:
# ======================================================================
#  4b — CWT 2D Morlet anisotrope sur zeta(z,x)
#  ----------------------------------------------------------------------
#  Morlet 2D dans Fourier (séparable, périodique en x, "réfléchie" en z) :
#     psî(kx,kz) = (analytique) * exp[-0.5 σx0²(s_x kx - k0)²] * exp[-0.5 σz0²(s_z kz)²]
#  -> 4 params balayés : (b_x,b_z) = TOUT le plan (sortie complète),
#                        (λ_x,λ_z) = grilles d'échelles en x ET en z.
#  Convention ω0=6, fourier_factor identique au 1D (cellules 21/22).
#  Sortie compacte : énergie 2D E(λx,λz) marginalisée + carte (x,z) à l'échelle pic.
# ======================================================================
import numpy as np

W0_MORLET = 6.0
NSX, NSZ  = 12, 10                       # nb d'échelles en x et en z (balayage 2D)

Lx_dom = n_x * dxl
Lz_dom = (z_D[-1] - z_D[0])

# --- grilles d'échelles : ~2 mailles jusqu'à ~moitié du domaine, en log ---
scales_x = np.geomspace(2*dxl, Lx_dom/2, NSX)
scales_z = np.geomspace(2*dzl, Lz_dom/2, NSZ)
fourier_factor = (4*np.pi) / (W0_MORLET + np.sqrt(2 + W0_MORLET**2))
lamx_km = scales_x * fourier_factor / 1000.0
lamz_km = scales_z * fourier_factor / 1000.0

# --- on travaille sur une grille z RÉGULARISÉE (FFT en z propre) ---
z_reg = np.linspace(z_D[0], z_D[-1], nz_D)
zeta_reg = np.empty_like(zeta)
for i in range(n_x):
    zeta_reg[:, i] = np.interp(z_reg, z_D, zeta[:, i])
dzr = float(z_reg[1]-z_reg[0])

kx = np.fft.fftfreq(n_x,  d=dxl) * 2*np.pi      # [rad/m]
kz = np.fft.fftfreq(nz_D, d=dzr) * 2*np.pi
KX, KZ = np.meshgrid(kx, kz)                    # (nz_D, n_x)
Zhat = np.fft.fft2(zeta_reg)

k0 = W0_MORLET                                  # pic de la porteuse (en unités s·k)

def cwt2d_morlet(sx, sz):
    """Coefficient CWT 2D pour l'échelle (sx,sz). Retourne |W|² (nz_D,n_x)."""
    skx = sx * KX
    skz = sz * KZ
    # porteuse le long de x (analytique : ne garde que skx>0), enveloppe en z
    psih = (np.pi**-0.5) * np.sqrt(sx*sz/(dxl*dzr)) \
           * (skx > 0) * np.exp(-0.5*((skx - k0)**2 + skz**2))
    W = np.fft.ifft2(Zhat * psih)
    return np.abs(W)**2

# --- énergie 2D E(λx,λz) = somme sur (x,z) de |W|² ; + carte au pic ---
E2 = np.zeros((NSZ, NSX))
for a in range(NSZ):
    for b in range(NSX):
        E2[a, b] = cwt2d_morlet(scales_x[b], scales_z[a]).sum()

az, bx = np.unravel_index(np.argmax(E2), E2.shape)
sx_pk, sz_pk = scales_x[bx], scales_z[az]
print(f"échelle 2D dominante : λx ≈ {lamx_km[bx]:.0f} km,  λz ≈ {lamz_km[az]:.1f} km")

# carte de présence (x,z) à l'échelle pic -> "où" est la structure dominante
Wmap_pk = cwt2d_morlet(sx_pk, sz_pk)            # (nz_D, n_x)

# ======================================================================
#  FIGURE — (a) spectre 2D d'échelle E(λx,λz)   (b) carte (x,z) à l'échelle pic
# ======================================================================
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))

pc0 = ax[0].pcolormesh(lamx_km, lamz_km, E2, shading='auto', cmap='magma')
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].plot(lamx_km[bx], lamz_km[az], 'c*', ms=16, mec='k', label='pic')
ax[0].set_xlabel(r'$\lambda_x$ [km]'); ax[0].set_ylabel(r'$\lambda_z$ [km]')
ax[0].set_title('(a) Énergie 2D des ondelettes $E(\\lambda_x,\\lambda_z)$')
ax[0].legend(fontsize=9); fig.colorbar(pc0, ax=ax[0], label='énergie', pad=0.01)

pc1 = ax[1].pcolormesh(x/1000, z_reg/1000, Wmap_pk, shading='auto', cmap='magma')
ax[1].set_xlabel('x [km]'); ax[1].set_ylabel('z [km]')
ax[1].set_title(fr'(b) Présence à $(\lambda_x,\lambda_z)\approx({lamx_km[bx]:.0f},{lamz_km[az]:.1f})$ km')
fig.colorbar(pc1, ax=ax[1], label=r'$|W_{2D}|^2$', pad=0.01)
plt.tight_layout(); plt.show()

---
# 5. Extraction des paquets d'ondes → localiser & cerner les tourbillons

> *« On est plus intéressé par les caractéristiques des tourbillons (qu'on doit d'abord
> localiser et cerner). »*

La carte $|W_{2D}|^2$ à l'échelle dominante donne **où** la circulation est intense, mais pas
encore des **objets** discrets. On en extrait des **paquets** : régions connexes où l'énergie
ondelette dépasse un seuil. Chaque paquet = un **tourbillon candidat** qu'on cerne par sa boîte
englobante, son barycentre, son aire, et le **signe de $\zeta$** en son cœur (sens de rotation).

In [ ]:
# ======================================================================
#  5a — Paquets d'ondes : seuillage de |W_2D|² à l'échelle pic -> objets
#  Réutilise Wmap_pk, zeta_reg, psi2d, x, z_reg. Pas de dépendance externe
#  lourde : labeling maison (flood-fill BFS) -> aucune install requise.
# ======================================================================
import numpy as np
from collections import deque

# --- normalisation + seuil (quantile haut de l'énergie) ---
Wn = Wmap_pk / (Wmap_pk.max() + 1e-30)
SEUIL = np.quantile(Wn, 0.92)            # 8% les plus énergétiques
mask = Wn >= SEUIL
nzr, nxr = mask.shape

# --- labeling 4-connexe, PÉRIODIQUE en x (wrap colonnes), borné en z ---
labels = np.zeros_like(mask, dtype=int)
cur = 0
for j0 in range(nzr):
    for i0 in range(nxr):
        if mask[j0, i0] and labels[j0, i0] == 0:
            cur += 1
            q = deque([(j0, i0)]); labels[j0, i0] = cur
            while q:
                j, i = q.popleft()
                vois = [(j+1, i), (j-1, i), (j, (i+1) % nxr), (j, (i-1) % nxr)]
                for jj, ii in vois:
                    if 0 <= jj < nzr and mask[jj, ii] and labels[jj, ii] == 0:
                        labels[jj, ii] = cur; q.append((jj, ii))

# --- filtre taille mini + mesure de chaque objet ---
AIRE_MIN = max(6, int(0.0015 * nzr * nxr))
vortices = []
for lab in range(1, cur+1):
    sel = labels == lab
    if sel.sum() < AIRE_MIN:
        labels[sel] = 0
        continue
    jj, ii = np.where(sel)
    Wsel = Wmap_pk[sel]
    # barycentre pondéré énergie
    jc = np.average(jj, weights=Wsel); ic = np.average(ii, weights=Wsel)
    jcc, icc = int(round(jc)), int(round(ic)) % nxr
    vortices.append(dict(
        label=lab, aire=int(sel.sum()),
        xc_km=x[icc]/1000, zc_km=z_reg[jcc]/1000,
        x_span_km=(ii.max()-ii.min())*dxl/1000,
        z_span_km=(jj.max()-jj.min())*dzr/1000,
        zeta_core=float(zeta_reg[jcc, icc]),
        sens=('cyclonique +' if zeta_reg[jcc, icc] > 0 else 'anticyclonique −'),
        E=float(Wsel.sum()),
        jc=jcc, ic=icc,
    ))

vortices.sort(key=lambda d: -d['E'])
n_vor = len(vortices)
print(f"{n_vor} tourbillon(s) extrait(s)  (seuil q=0.92, aire_min={AIRE_MIN} px)\n")
print(f"{'#':>2} | {'x_c':>6} | {'z_c':>5} | {'Δx':>5} | {'Δz':>5} | {'ζ_cœur':>9} | {'sens':>14}")
print('-'*64)
for r, v in enumerate(vortices, 1):
    print(f"{r:>2} | {v['xc_km']:6.0f} | {v['zc_km']:5.1f} | {v['x_span_km']:5.0f} | "
          f"{v['z_span_km']:5.1f} | {v['zeta_core']:+9.2e} | {v['sens']:>14}")

# ======================================================================
#  FIGURE — paquets sur fond zeta + isolignes psi + boîtes englobantes
# ======================================================================
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(13, 5.5))
zmax = np.percentile(np.abs(zeta_reg), 99) + 1e-30
pc = ax.pcolormesh(x/1000, z_reg/1000, zeta_reg, shading='auto',
                   norm=TwoSlopeNorm(vcenter=0, vmin=-zmax, vmax=zmax), cmap='RdBu_r')
# psi interpolé sur z_reg pour superposition cohérente
psi_reg = np.empty_like(zeta_reg)
for i in range(nxr):
    psi_reg[:, i] = np.interp(z_reg, z_D, psi2d[:, i])
ax.contour(x/1000, z_reg/1000, psi_reg, levels=20, colors='k', linewidths=0.5, alpha=0.6)

# contours des paquets + boîtes + numéro
ax.contour(x/1000, z_reg/1000, (labels > 0).astype(float), levels=[0.5],
           colors='lime', linewidths=1.6)
for r, v in enumerate(vortices, 1):
    sel = labels == v['label']; jj, ii = np.where(sel)
    x0, x1 = x[ii.min()]/1000, x[ii.max()]/1000
    z0, z1 = z_reg[jj.min()]/1000, z_reg[jj.max()]/1000
    ax.add_patch(Rectangle((x0, z0), x1-x0, z1-z0, fill=False, ec='lime', lw=1.0, ls='--'))
    ax.text(v['xc_km'], v['zc_km'], str(r), color='k', fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='circle,pad=0.15', fc='lime', ec='k', alpha=0.85))

ax.set_xlabel('x [km]'); ax.set_ylabel('z [km]')
ax.set_title(r'Tourbillons extraits — fond $\zeta$, isolignes $\psi$, paquets (vert)')
fig.colorbar(pc, ax=ax, label=r'$\zeta=\partial_x\bar w-\partial_z\bar u$ [s$^{-1}$]', pad=0.01)
plt.tight_layout(); plt.show()

---
# 6. Caractériser le « gros tourbillon » : échelle, $\psi$, symétrie $x/z$, orientation

> *« toutes les caractéristiques de mon gros tourbillon (associé au paquet d'ondelettes),
> orientation »* — *« mesure de la force de symétrie en $x$ et en $z$ »*.

On prend le **tourbillon dominant** (#1, le plus énergétique) et on en extrait une fiche
signalétique complète, mesurée sur la fenêtre du paquet :

- **échelles** $\lambda_x,\lambda_z$ déjà fournies par l'ondelette pic (4e param) ;
- **$\psi$ associé** : amplitude de circulation $\Delta\psi$ sur la fenêtre (débit de masse) ;
- **force de symétrie** en $x$ et en $z$ : corrélation du motif avec son miroir,
  $S=\dfrac{\langle f\cdot f_{\text{mir}}\rangle}{\langle f^2\rangle}\in[-1,1]$
  ($+1$ symétrique, $-1$ antisymétrique) ;
- **orientation** : axe principal du motif $|W|^2$ via les moments d'inertie (angle / horizontale).

In [ ]:
# ======================================================================
#  6a — Fiche signalétique du tourbillon dominant (#1)
#  Mesures sur la fenêtre du paquet : échelle (ondelette), Δψ, symétrie x/z,
#  orientation (axe principal des moments). Réutilise vortices, labels,
#  zeta_reg, psi_reg, Wmap_pk, lamx_km, lamz_km, bx, az, x, z_reg.
# ======================================================================
import numpy as np

assert len(vortices) > 0, "Aucun tourbillon extrait : baisse SEUIL en 5a."
v = vortices[0]                                   # le plus énergétique
sel = labels == v['label']; jj, ii = np.where(sel)
j0, j1 = jj.min(), jj.max(); i0, i1 = ii.min(), ii.max()

# --- fenêtre (boîte englobante) sur zeta et psi ---
Zwin   = zeta_reg[j0:j1+1, i0:i1+1]
Pwin   = psi_reg[j0:j1+1,  i0:i1+1]
Wwin   = Wmap_pk[j0:j1+1,  i0:i1+1]
xwin   = x[i0:i1+1]/1000; zwin = z_reg[j0:j1+1]/1000

# --- (1) échelles (4e paramètre de l'ondelette dominante) ---
lam_x, lam_z = lamx_km[bx], lamz_km[az]
aspect = lam_x / max(lam_z, 1e-9)

# --- (2) psi associé : amplitude de circulation (débit de masse) ---
dpsi = float(np.nanmax(Pwin) - np.nanmin(Pwin))   # [kg m^-1 s^-1]

# --- (3) force de symétrie : corrélation au miroir, sur zeta centré ---
def sym_force(f, axis):
    f = f - f.mean()
    fm = np.flip(f, axis=axis)
    num = np.sum(f*fm); den = np.sum(f*f) + 1e-30
    return float(num/den)                          # +1 sym, -1 antisym
S_x = sym_force(Zwin, axis=1)                      # miroir gauche-droite
S_z = sym_force(Zwin, axis=0)                      # miroir bas-haut

# --- (4) orientation : axe principal de |W|² (moments d'inertie 2D) ---
ww = Wwin / (Wwin.sum() + 1e-30)
ZZ, XX = np.meshgrid(np.arange(Wwin.shape[0])*dzr,
                     np.arange(Wwin.shape[1])*dxl, indexing='ij')
xc = np.sum(ww*XX); zc = np.sum(ww*ZZ)
xx = XX - xc; zz = ZZ - zc
Ixx = np.sum(ww*xx*xx); Izz = np.sum(ww*zz*zz); Ixz = np.sum(ww*xx*zz)
theta = 0.5*np.arctan2(2*Ixz, Ixx - Izz)           # axe principal / horizontale
theta_deg = np.degrees(theta)

# --- récap imprimé ---
print("="*58)
print(f"  FICHE — TOURBILLON DOMINANT (#1)")
print("="*58)
print(f"  position cœur ......... x={v['xc_km']:.0f} km , z={v['zc_km']:.1f} km")
print(f"  extension ............. Δx={v['x_span_km']:.0f} km , Δz={v['z_span_km']:.1f} km")
print(f"  échelle ondelette ..... λx={lam_x:.0f} km , λz={lam_z:.1f} km  (aspect {aspect:.1f})")
print(f"  sens de rotation ...... {v['sens']}   (ζ_cœur={v['zeta_core']:+.2e} s⁻¹)")
print(f"  circulation ........... Δψ={dpsi:.2e} kg m⁻¹ s⁻¹")
print(f"  symétrie en x ......... S_x={S_x:+.2f}   ({'symétrique' if S_x>0.3 else 'antisymétrique' if S_x<-0.3 else 'ni l’un ni l’autre'})")
print(f"  symétrie en z ......... S_z={S_z:+.2f}   ({'symétrique' if S_z>0.3 else 'antisymétrique' if S_z<-0.3 else 'ni l’un ni l’autre'})")
print(f"  orientation ........... θ={theta_deg:+.0f}° / horizontale")
print("="*58)

# ======================================================================
#  FIGURE — zoom sur le gros tourbillon : zeta + psi + axe principal
# ======================================================================
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

zmax = np.percentile(np.abs(Zwin), 99) + 1e-30
pc0 = ax[0].pcolormesh(xwin, zwin, Zwin, shading='auto',
                       norm=TwoSlopeNorm(vcenter=0, vmin=-zmax, vmax=zmax), cmap='RdBu_r')
ax[0].contour(xwin, zwin, Pwin, levels=14, colors='k', linewidths=0.6)
# axe principal (orientation) tracé depuis le barycentre
L = 0.4*max((xwin[-1]-xwin[0]), (zwin[-1]-zwin[0]))
xc_km = xwin[0] + xc/1000; zc_km = zwin[0] + zc/1000
ax[0].plot([xc_km - L*np.cos(theta), xc_km + L*np.cos(theta)],
           [zc_km - L*np.sin(theta), zc_km + L*np.sin(theta)],
           'lime', lw=2.2, label=f'axe principal θ={theta_deg:+.0f}°')
ax[0].set_xlabel('x [km]'); ax[0].set_ylabel('z [km]')
ax[0].set_title('(a) Cœur : ζ + isolignes ψ + orientation')
ax[0].legend(fontsize=9, loc='upper right')
fig.colorbar(pc0, ax=ax[0], label=r'$\zeta$ [s$^{-1}$]', pad=0.01)

wmax = np.percentile(Wwin, 99) + 1e-30
pc1 = ax[1].pcolormesh(xwin, zwin, Wwin, shading='auto', cmap='magma', vmax=wmax)
ax[1].set_xlabel('x [km]'); ax[1].set_ylabel('z [km]')
ax[1].set_title(r'(b) Énergie ondelette $|W_{2D}|^2$ du paquet')
fig.colorbar(pc1, ax=ax[1], label=r'$|W_{2D}|^2$', pad=0.01)
plt.tight_layout(); plt.show()

In [ ]:
# ======================================================================
#  6b — "Plusieurs endroits semblables" : on compare chaque tourbillon
#       au gabarit du #1 (corrélation de motif ζ, redimensionné).
#       -> classe les structures par ressemblance au gros tourbillon.
#  Réutilise vortices, labels, zeta_reg.
# ======================================================================
import numpy as np

def patch_norm(arr, ny=24, nx=24):
    """Rééchantillonne un patch en (ny,nx) par interpolation bilinéaire simple, centré-réduit."""
    a = np.asarray(arr, float)
    H, W = a.shape
    yi = np.linspace(0, H-1, ny); xi = np.linspace(0, W-1, nx)
    Y0 = np.floor(yi).astype(int); X0 = np.floor(xi).astype(int)
    Y1 = np.minimum(Y0+1, H-1);    X1 = np.minimum(X0+1, W-1)
    fy = (yi-Y0)[:, None]; fx = (xi-X0)[None, :]
    a00 = a[np.ix_(Y0, X0)]; a01 = a[np.ix_(Y0, X1)]
    a10 = a[np.ix_(Y1, X0)]; a11 = a[np.ix_(Y1, X1)]
    out = (a00*(1-fy)*(1-fx) + a01*(1-fy)*fx + a10*fy*(1-fx) + a11*fy*fx)
    out = out - out.mean()
    return out / (np.linalg.norm(out) + 1e-30)

# gabarit = patch ζ du tourbillon #1
def patch_of(v):
    sel = labels == v['label']; jj, ii = np.where(sel)
    return zeta_reg[jj.min():jj.max()+1, ii.min():ii.max()+1]

g = patch_norm(patch_of(vortices[0]))
print(f"{'#':>2} | {'x_c':>6} | {'z_c':>5} | {'corr vs #1':>11} | {'sens':>14}")
print('-'*52)
sims = []
for r, v in enumerate(vortices, 1):
    p = patch_norm(patch_of(v))
    corr = float(np.sum(g*p))                       # cos-sim (patches normés)
    sims.append(corr)
    tag = ' (référence)' if r == 1 else ''
    print(f"{r:>2} | {v['xc_km']:6.0f} | {v['zc_km']:5.1f} | {corr:+11.2f} | {v['sens']:>14}{tag}")

sims = np.array(sims)
seuil_sem = 0.6
idx_sem = np.where(sims >= seuil_sem)[0]
print(f"\n>> {len(idx_sem)} structure(s) « semblable(s) » au gros tourbillon "
      f"(corr ≥ {seuil_sem}) : #{', #'.join(str(i+1) for i in idx_sem)}")
print(">> interprétation : ces endroits partagent la même signature (rouleau / overturning),")
print(">>   candidats pour une statistique de population (échelle, Δψ, orientation moyennes).")

### Bilan de l'étape ondelettes 2D → tourbillons

Ce bloc répond aux trois flèches de la feuille de route :

1. **Vérif du code 1D Morlet** — la CWT 2D réutilise *exactement* la même normalisation
   énergie et le même `fourier_factor` que les cellules 21/22 ; le 1D-en-$x$ est le cas
   $\lambda_z\to\infty$ de ce 2D, ce qui sert de contrôle de cohérence.
2. **Ondelettes sans fixer $z$, 4 params** — $(b_x,b_z,\lambda_x,\lambda_z)$ explicitement
   balayés ; la « meilleure décompo » pour un rouleau est l'anisotrope ($\lambda_x\!\neq\!\lambda_z$),
   confirmée par le pic du spectre 2D $E(\lambda_x,\lambda_z)$.
3. **Paquets → tourbillons cernés** — extraction d'objets, fiche signalétique
   (échelle, $\Delta\psi$, symétrie $x/z$, orientation), et recensement des
   **endroits semblables** pour passer à une statistique de population.

**Lien paramétrisation CMT.** L'échelle dominante $\lambda_x$ et la circulation $\Delta\psi$
des tourbillons donnent une longueur et une intensité *mesurées* — directement injectables
dans la fermeture de $T_2$ comme amplitude $r_0\rho_0\sigma_{up}\sigma_u\sigma_w$, au lieu d'un
$C_\sigma$ ad hoc. C'est l'étape qui relie la géométrie des rouleaux au flux organisé/résiduel.